# CEADS parser walkthrough

This notebook is the practical guide for parsing the currently supported CEADS China provincial MRIO workbooks in MARIO.

## What this notebook covers

- where the supported CEADS workbooks come from;
- which workbook family MARIO currently parses;
- what `format=` means;
- how to use `path=` and `year=`;
- how MARIO maps imports, exports, and the optional `CO2` row.

## Source pages

- CEADS portal: [Input-Output Tables](https://www.ceads.net/data/input_output_tables/)
- Scientific Data descriptor: [China's Provincial Multi-Regional Input-Output Database for 2018 and 2020](https://www.nature.com/articles/s41597-025-06543-y)
- Dataset DOI: [10.6084/m9.figshare.29927291](https://doi.org/10.6084/m9.figshare.29927291)

MARIO does not download the CEADS workbook automatically. Download the workbook locally before running this notebook.

## Main entry point

For normal user workflows, the public entry point is `mario.parse_ceads(...)`.

At the moment, that entry point supports only `table="IOT"`.

## What MARIO currently parses

The current backend is intentionally narrow:

- it parses local Excel workbooks only;
- it currently supports the verified CEADS provincial MRIO workbook family for `2018` and `2020`;
- it reads the English table sheet, not the Chinese one;
- it uses the `Sector` and `Province` sheets as metadata support.

## What `format` means

`format=` is the parser-side layout selector. It does not mean "which CEADS dataset" you are using; it means "which workbook structure" MARIO should expect.

At the moment, MARIO supports:

- `format="auto"`: inspect the workbook and resolve the known layout automatically;
- `format="ceads_provincial_workbook"`: force the verified 2018/2020 workbook structure.

## Configure local paths

Set `MARIO_DOC_DATA` to the root folder where you keep real parser datasets, or set `MARIO_CEADS_PATH` directly to the CEADS folder.

Expected local structure:

```text
CEADS/
├── MRIO 2018.xlsx
└── MRIO 2020.xlsx
```

In [1]:
from pathlib import Path
import os

import mario

## Parse one explicit workbook

Use this form when you want to parse a specific downloaded workbook.

In [2]:
db_2018 = mario.parse_ceads(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/CEADS/MRIO 2018.xlsx',
    format="auto",
    calc_all=False,
)

db_2018

INFO Parser: inspecting CEADS workbook MRIO 2018.xlsx.


INFO Parser: detected CEADS workbook format ceads_provincial_workbook for year 2018 using sheet Table_2018_English Version.


INFO Parser: reading CEADS workbook MRIO 2018.xlsx sheet Table_2018_English Version.


INFO Parser: CEADS payload ready with shapes Z=(1302, 1302), Y=(1302, 186), V=(5, 1302), E=(1, 1302), EY=(1, 186).


INFO Metadata: initialized.


name = CEADS China Provincial MRIO 2018
table = IOT
scenarios = ['baseline']
Factor of production = 5
Satellite account = 1
Consumption category = 6
Region = 31
Sector = 42

## Parse from a directory

Use this form when one folder contains multiple CEADS workbooks. In that case, `year=` selects which workbook to parse.

In [3]:
db_2020 = mario.parse_ceads(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/CEADS/MRIO 2020.xlsx',
    year=2020,
    calc_all=False,
)

db_2020

INFO Parser: inspecting CEADS workbook MRIO 2020.xlsx.


INFO Parser: detected CEADS workbook format ceads_provincial_workbook for year 2020 using sheet Table_2020_English Version.


INFO Parser: reading CEADS workbook MRIO 2020.xlsx sheet Table_2020_English Version.


INFO Parser: CEADS payload ready with shapes Z=(1302, 1302), Y=(1302, 186), V=(5, 1302), E=(1, 1302), EY=(1, 186).


INFO Metadata: initialized.


name = CEADS China Provincial MRIO 2020
table = IOT
scenarios = ['baseline']
Factor of production = 5
Satellite account = 1
Consumption category = 6
Region = 31
Sector = 42

## How the parsed table is mapped

For the verified CEADS workbook family:

- `Z` comes from the main intermediate-transactions block;
- `Y` contains the domestic final-demand block plus one `Exports` column per province;
- `V` contains `Imports` plus the four value-added rows exposed by the workbook;
- `E` contains the optional `CO2 (Mt)` row when it exists;
- `EY` is initialized at zero.

In [4]:
db_2020["baseline"].keys()

dict_keys(['Z', 'Y', 'V', 'E', 'EY'])

## inspect the parsed database

Once parsed, the result is a standard MARIO database.

In [5]:
db_2020.get_index("Region")

['Beijing',
 'Tianjin',
 'Hebei',
 'Shanxi',
 'Inner Mongolia',
 'Liaoning',
 'Jilin',
 'Heilongjiang',
 'Shanghai',
 'Jiangsu',
 'Zhejiang',
 'Anhui',
 'Fujian',
 'Jiangxi',
 'Shandong',
 'He0',
 'Hubei',
 'Hu0',
 'Guangdong',
 'Guangxi',
 'Hai0',
 'Chongqing',
 'Sichuan',
 'Guizhou',
 'Yun0',
 'Xizang',
 'Shaanxi',
 'Gansu',
 'Qinghai',
 'Ningxia',
 'Xinjiang']

In [6]:
db_2020.get_index("Sector")

['Agriculture, Forestry, Animal Husbandry and Fishery',
 'Mining and washing of coal',
 'Extraction of petroleum and natural gas',
 'Mining and processing of metal ores',
 'Mining and processing of nonmetal and other ores',
 'Food and tobacco processing',
 'Textile industry',
 'Manufacture of leather, fur, feather and related products',
 'Processing of timber and furniture',
 'Manufacture of paper, printing and articles for culture, education and sport activity',
 'Processing of petroleum, coking, processing of nuclear fuel',
 'Manufacture of chemical products',
 'Manuf. of non -metallic mineral products',
 'Smelting and processing of metals',
 'Manufacture of metal products',
 'Manufacture of general purpose machinery',
 'Manufacture of special purpose machinery',
 'Manufacture of transport equipment',
 'Manufacture of electrical machinery and equipment',
 'Manufacture of communication equipment, computers and other electronic equipment',
 'Manufacture of measuring instruments',
 'Oth

In [7]:
db_2020.get_index("Satellite account")

['CO2']

## Caveats

- The CEADS portal lists older provincial MRIO releases too, such as `2012`, `2015`, and `2017`. MARIO does not assume those older workbook families are compatible until they are checked explicitly.
- The parser intentionally relies on the English worksheet. If only a Chinese worksheet is available, that workbook is not currently supported.
- The workbook metadata sheets can contain minor naming inconsistencies. For the verified 2018/2020 family, MARIO treats the main transaction axis as canonical when such conflicts appear.